# 使用 Oracle AI Agent Memory 为智能体启用长期记忆

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/openai/openai-cookbook/blob/main/examples/vector_databases/oracle_db/deep_research_openai_agents.ipynb) [![文档](https://img.shields.io/badge/Documentation-Oracle%20AI%20Agent%20Memory-red?style=flat-square)](https://www.oracle.com/database/ai-agent-memory/)

**框架：** OpenAI Agents SDK | **网页搜索：** Tavily | **记忆：** 基于 Oracle AI Database 的 Oracle AI Agent Memory

长时间运行的智能体工作流往往不只需要短期的对话回放。它们需要一种方法，能够跨会话记住持久的发现、决策和可复用的上下文，这样智能体就能接着之前的工作继续，而不必每次都从头开始。

本 notebook 演示了一种**面向 OpenAI Agents SDK、以 Oracle 为后端的 Session 模式**。它并不是通用的 OpenAI 记忆实现。Agents SDK 的 `Session` 接口为当前智能体循环存储短期对话历史；持久的研究发现则通过以 Oracle AI Agent Memory 为后端的显式记忆工具单独保存。

本示例使用一个用于人类基因组探索的**深度研究智能体**，但核心模式的适用范围更广：将 OpenAI Agents SDK 的会话持久化与用于可复用发现的持久记忆层结合起来。该智能体使用 Tavily 进行实时网页研究，将 OpenAI Agents SDK 的会话条目存储在 Oracle AI Database 中，并通过专门的 `save_research_finding` 工具保存持久发现。

## 为什么长期记忆很重要

短期会话记忆帮助智能体保持当前对话的连贯性。长期记忆解决的是另一个问题：它让智能体能够跨会话保留选定的发现、恢复之前的工作，并避免重新去发现已经研究过的事实。

在本例中，会话条目和持久发现具有不同的生命周期。会话条目是在智能体循环内用于回放的原始对话状态。持久发现则是经过筛选、可搜索的结论，由智能体通过工具有意保存。

当应用本身已经运行在 Oracle AI Database 上，并且需要受管控的持久记忆、对已保存发现的语义搜索，以及在同一个数据库底座上按作用域访问短期会话回放和长期记忆时，Oracle AI Agent Memory 就很适合这种模式。

---

## 智能体应用中的运行模式

AI 智能体应用通常分为三种运行模式：

| 模式 | 形态 | 典型用途 |
|---|---|---|
| **助手（Assistant）** | 逐轮对话式 | 客户支持、编程助手、聊天界面 |
| **深度研究（Deep Research）** | 多步骤自主调查 | 文献综述、市场研究、技术范围界定 |
| **工作流（Workflow）** | 预先确定的步骤序列，通常带有条件分支 | 审批流水线、合规检查、结构化分诊 |

> **注意：** 本 notebook 聚焦于深度研究模式。
>
> 深度研究智能体会在长周期的调查中进行规划、检索、综合并积累发现。记忆是核心，但并非所有记忆都有相同的生命周期：会话条目支持短期的连续性，而持久发现是智能体可以跨会话复用的经过筛选的事实。


## 你将学到什么

- 如何基于自定义后端（Oracle AI Agent Memory）实现 **OpenAI Agents SDK 的 `Session` 协议**
- 如何将 **Tavily** 封装为智能体可调用的 `function_tool`
- 如何将长期有效的研究发现存储为持久的**记忆**（与短期对话历史分开）
- 如何验证跨会话的记忆连续性——智能体可以从上次中断的地方继续

> **💡 关键洞察：** 短期记忆（当前运行的对话历史）和长期记忆（跨运行的持久发现）是对同一存储的不同访问模式。前者我们使用 `Session`，后者使用 `add_memory()`。

## 前提条件

- 本 notebook 推荐使用 **Python 3.10 或 3.11**
- 本地运行（Docker 容器）或可通过网络访问的 **Oracle AI Database**
- 供智能体的 LLM 和嵌入使用的 **`OPENAI_API_KEY`**
- 用于网页搜索的 **`TAVILY_API_KEY`**（可在 [tavily.com](https://tavily.com) 获取免费套餐）
- 在当前内核环境中安装 `oracleagentmemory`、`litellm`、`openai-agents`、`tavily-python`、`python-dotenv` 和 `nest_asyncio` 包

如果你在 Colab 中运行本 notebook，Oracle 数据库必须能够从 Colab 运行时访问。Colab 无法通过 `localhost` 访问你笔记本电脑上的 Docker 数据库；请通过可访问的主机/网络路径暴露数据库，或者在 Docker 容器所在的机器上本地运行 notebook。

对于暴露了 `1521` 端口的本地 Oracle Database Free Docker 容器，DSN 通常形如 `localhost:1521/FREEPDB1`。如果你的容器将 Oracle 监听器映射到了不同的主机端口，请在 `ORACLE_DSN` 中使用该主机端口。在运行连接单元格之前，请等待容器报告 `DATABASE IS READY TO USE!`。

在 notebook 旁边创建一个 `.env` 文件，或在 shell 中设置以下变量：

```env
OPENAI_API_KEY=
TAVILY_API_KEY=
ORACLE_USER=
ORACLE_PASSWORD=
ORACLE_DSN=
# Optional: set OPENAI_MODEL to use a different OpenAI model
# OPENAI_MODEL=
```

本 notebook 也接受旧版别名 `DB_USER`、`DB_PASSWORD` 和 `DB_CONNECT_STRING`，但推荐使用上面的 Oracle 风格名称。


## 1. 安装依赖

我们安装 notebook 所需的依赖：`oracleagentmemory` 用于以 Oracle 为后端的记忆，`litellm` 用于 Oracle AI Agent Memory 所使用的嵌入路径，`openai-agents` 用于 Agents SDK 运行时，`tavily-python` 用于网页搜索，`python-dotenv` 用于环境变量，`nest_asyncio` 则让 `Runner.run` 能在 Jupyter 事件循环中顺利运行。

依赖边界：`openai-agents` 提供 Agents SDK 运行时和会话协议，而 `litellm` 由 Oracle AI Agent Memory 用于嵌入以及对持久发现的语义搜索。普通的 Agents SDK 会话实现并不需要 Oracle 的长期记忆路径。


In [1]:
%%capture
%pip install -q oracleagentmemory litellm openai-agents tavily-python nest_asyncio python-dotenv

## 2. 配置 API 密钥和 Oracle 连接

从 `.env` 或环境变量加载配置。本 notebook 有意**不**提供默认凭据。如果缺少必需的值，它会提示你输入，而不是将密钥嵌入 notebook 中。

必需的值：

- `OPENAI_API_KEY`
- `TAVILY_API_KEY`
- `ORACLE_USER` 或 `DB_USER`
- `ORACLE_PASSWORD` 或 `DB_PASSWORD`
- `ORACLE_DSN` 或 `DB_CONNECT_STRING`

In [2]:
import os
import getpass
from pathlib import Path

from dotenv import load_dotenv

# Load .env from the current working directory or notebook directory.
load_dotenv()
load_dotenv(Path.cwd() / ".env")


def _read_setting(primary: str, *aliases: str, secret: bool = False) -> str:
    """Read a required setting from env vars, falling back to a notebook prompt."""
    for key in (primary, *aliases):
        value = os.getenv(key)
        if value:
            os.environ[primary] = value
            return value

    prompt = f"Enter {primary}: "
    value = getpass.getpass(prompt) if secret else input(prompt)
    if not value:
        raise RuntimeError(f"Missing required setting: {primary}")
    os.environ[primary] = value
    return value


OPENAI_API_KEY = _read_setting("OPENAI_API_KEY", secret=True)
TAVILY_API_KEY = _read_setting("TAVILY_API_KEY", secret=True)
DB_USER = _read_setting("DB_USER", "ORACLE_USER")
DB_PASSWORD = _read_setting("DB_PASSWORD", "ORACLE_PASSWORD", secret=True)
DB_CONNECT_STRING = _read_setting("DB_CONNECT_STRING", "ORACLE_DSN")
MODEL = os.getenv("OPENAI_MODEL") or "gpt-5.4"

# Keep later cells consistent if a notebook user updates these Python variables directly.
os.environ.update({
    "OPENAI_API_KEY": OPENAI_API_KEY,
    "TAVILY_API_KEY": TAVILY_API_KEY,
    "DB_USER": DB_USER,
    "DB_PASSWORD": DB_PASSWORD,
    "DB_CONNECT_STRING": DB_CONNECT_STRING,
})

# Jupyter runs an async event loop already; this lets us call async SDK methods cleanly.
import nest_asyncio
nest_asyncio.apply()

print("Configuration loaded from environment variables or prompts.")
print("OpenAI model configured.")


Configuration loaded from environment variables or prompts.
OpenAI model configured.


## 3. 连接 Oracle AI Database 并初始化记忆客户端

`OracleAgentMemory` 是受管控的记忆客户端。在本 notebook 中，自动提取功能被禁用，以便明确区分：

- **会话条目**是用于回放的短期 OpenAI Agents SDK 对话状态。
- **研究发现**是仅通过 `save_research_finding` 工具写入的持久记忆。

我们使用 `text-embedding-3-small` 对持久发现进行语义搜索。

> **💡 关键洞察：** `schema_policy` 控制库在启动时如何与数据库交互。`REQUIRE_EXISTING` 从不执行 DDL。`CREATE_IF_NECESSARY` 会补全缺失的对象，对演示和开发而言更安全。`RECREATE` 会删除并重建对象，不应作为共享 schema 中的默认设置。

In [3]:
import oracledb
from oracleagentmemory.core import OracleAgentMemory
from oracleagentmemory.core.dbschemapolicy import SchemaPolicy

connection = oracledb.connect(
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    dsn=os.environ["DB_CONNECT_STRING"],
)
print("Connected to Oracle AI Database.")

memory_client = OracleAgentMemory(
    connection=connection,
    embedder="text-embedding-3-small",   # OpenAI-compatible; 1536 dims
    extract_memories=False,               # durable findings are saved explicitly via tool calls
    schema_policy=SchemaPolicy.CREATE_IF_NECESSARY,
    table_name_prefix="genome_",         # isolates this example's tables from others
)
print("Memory client ready with safe schema policy:", SchemaPolicy.CREATE_IF_NECESSARY.value)

Connected to Oracle AI Database.
Memory client ready with safe schema policy: create_if_necessary


## 4. 注册研究用户和智能体

使用唯一的运行作用域，这样重新运行 notebook 时就不会把旧的演示记忆与当前运行混在一起。这既能保证 `CREATE_IF_NECESSARY` 的安全，又能避免引入之前执行留下的过时长期发现。

In [4]:
from datetime import datetime, timezone

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
USER_ID = f"genome-researcher-{RUN_ID}"
AGENT_ID = f"deep-research-agent-{RUN_ID}"

for create_fn, eid, info in [
    (memory_client.add_user, USER_ID, "Genomics researcher exploring human disease associations."),
    (memory_client.add_agent, AGENT_ID, "Deep-research agent with web search and long-term memory."),
]:
    try:
        create_fn(eid, info)
        print(f"Registered: {eid}")
    except ValueError:
        print(f"Already exists: {eid}")

print(f"Run scope: {RUN_ID}")

Registered: genome-researcher-20260715083601
Registered: deep-research-agent-20260715083601
Run scope: 20260715083601


## 5. 基于 Oracle AI Agent Memory 实现 `Session` 协议

OpenAI Agents SDK 的 `Session` 协议负责**短期会话持久化**：在智能体循环中，runner 会请求之前的条目并追加新条目。

在此实现中，会话条目以 Oracle Agent Memory 的 `message` 记录形式存储，并带有 `metadata={"session_item": true}`。它们按 `session_id`、`user_id` 和 `agent_id` 划分作用域，仅用于回放对话。持久的长期发现则通过 `save_research_finding` 工具单独存储为 `memory` 记录。

> **重要：** 原始的模型/工具消息可能体积大且噪声多。除非某个工具有意将其提炼为简洁的结论，否则它们不应作为长期研究发现出现。

In [5]:
import json
from typing import Optional
from agents.memory.session import SessionABC


class OracleAgentMemorySession(SessionABC):
    """Session backend that persists OpenAI Agents SDK items in Oracle AI Database.

    Session items are short-term replay state stored as message records. Durable
    research findings are saved separately via the save_research_finding tool.
    """

    def __init__(self, session_id: str, client: OracleAgentMemory, user_id: str, agent_id: str):
        self.session_id = session_id
        self._client = client
        self._store = client._store
        self._user_id = user_id
        self._agent_id = agent_id
        try:
            self._client.create_thread(
                thread_id=session_id,
                user_id=user_id,
                agent_id=agent_id,
                extract_memories=False,
            )
        except ValueError:
            pass  # thread already exists
        self._next_session_seq = self._load_next_session_seq()

    async def get_items(self, limit: Optional[int] = None) -> list:
        records = self._session_records()
        ordered = sorted(records, key=self._session_sort_key)
        if limit:
            ordered = ordered[-limit:]
        return [json.loads(r.content) for r in ordered]

    async def add_items(self, items: list) -> None:
        if not items:
            return
        start_seq = self._next_session_seq
        metadata = [
            {
                "session_item": True,
                "session_id": self.session_id,
                "session_seq": start_seq + offset,
            }
            for offset, _ in enumerate(items)
        ]
        self._store.add(
            contents=[json.dumps(item) for item in items],
            record_type="message",
            thread_ids=self.session_id,
            user_ids=self._user_id,
            agent_ids=self._agent_id,
            roles=[self._message_role(item) for item in items],
            metadata=metadata,
        )
        self._next_session_seq = start_seq + len(items)

    async def pop_item(self) -> Optional[dict]:
        records = self._session_records()
        if not records:
            return None
        last = max(records, key=self._session_sort_key)
        self._store.delete("message", last.id)
        return json.loads(last.content)

    async def clear_session(self) -> None:
        records = self._session_records()
        for r in records:
            self._store.delete("message", r.id)
        self._next_session_seq = 0

    def _session_records(self) -> list:
        return self._store.list(
            "message",
            thread_id=self.session_id,
            user_id=self._user_id,
            agent_id=self._agent_id,
            limit=10_000,
            metadata_filter={"session_item": True},
        )

    def _load_next_session_seq(self) -> int:
        seqs = [self._session_seq(r) for r in self._session_records()]
        seqs = [seq for seq in seqs if seq is not None]
        return max(seqs) + 1 if seqs else 0

    @staticmethod
    def _record_metadata(record) -> dict:
        metadata = getattr(record, "metadata", None) or {}
        if isinstance(metadata, str):
            try:
                return json.loads(metadata)
            except json.JSONDecodeError:
                return {}
        return metadata if isinstance(metadata, dict) else {}

    @classmethod
    def _session_seq(cls, record) -> Optional[int]:
        seq = cls._record_metadata(record).get("session_seq")
        try:
            return int(seq)
        except (TypeError, ValueError):
            return None

    @classmethod
    def _session_sort_key(cls, record) -> tuple:
        seq = cls._session_seq(record)
        if seq is not None:
            return (1, seq)
        # Fallback only covers records created before this notebook stored session_seq.
        created_at = getattr(record, "created_at", "") or ""
        return (0, str(created_at), str(getattr(record, "id", "")))

    @staticmethod
    def _message_role(item: dict) -> str:
        role = item.get("role") if isinstance(item, dict) else None
        if role in {"user", "assistant", "system", "tool"}:
            return role
        return "user"


## 6. 定义智能体的工具

该智能体需要三个工具：

1. **`tavily_search`**：实时网页搜索，获取最新的基因组学资料来源
2. **`save_research_finding`**：持久化一条研究笔记，供智能体在未来的会话中回忆
3. **`recall_research_findings`**：在长期记忆中搜索某个主题的已有发现

工具 2 和 3 是智能体管理其**长期**记忆的方式。它们与 `Session` 机制不同，后者会自动处理短期对话历史。

> **📌 `@function_tool` 装饰器**会根据 Python 类型注解自动生成工具的 JSON schema，并解析 docstring 作为 LLM 看到的描述。请编写信息充分的 docstring——LLM 会读取它们。

In [6]:
from agents import function_tool
from tavily import TavilyClient

_tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

@function_tool
def tavily_search(query: str, max_results: int = 5) -> str:
    """Search the live web for recent, authoritative genomics sources.

    Use this to pull current information from NCBI, OMIM, peer-reviewed journals,
    and other scientific sources. Prefer this over relying on model-internal knowledge
    for facts about specific genes, mutations, or recent studies.

    Args:
        query: Natural-language search query.
        max_results: How many results to return (1-10).
    """
    resp = _tavily.search(
        query=query, max_results=max_results,
        search_depth="advanced", include_answer=True,
    )
    lines = [f"Answer: {resp.get('answer', '')}"]
    for r in resp.get("results", []):
        lines.append(f"- {r['title']} ({r['url']})\n  {r['content'][:300]}")
    return "\n".join(lines)


@function_tool
def save_research_finding(topic: str, finding: str) -> str:
    """Persist a durable research finding the agent should recall in future sessions.

    Use this for claims worth remembering across sessions, for example: "BRCA1
    pathogenic variants are associated with substantially elevated lifetime breast
    cancer risk." Do not use this for conversational acknowledgements, raw tool
    output, or speculation.

    Args:
        topic: Short topic tag (e.g. 'BRCA1', 'TP53 mutations').
        finding: The finding to remember, ideally one sentence.
    """
    memory_id = memory_client.add_memory(
        f"[{topic}] {finding}",
        user_id=USER_ID,
        agent_id=AGENT_ID,
        metadata={"topic": topic, "kind": "research_finding", "durable": True},
    )
    return f"Saved durable research finding (id={memory_id})."


@function_tool
def recall_research_findings(query: str, max_results: int = 5) -> str:
    """Search prior durable research findings for information relevant to a query.

    Use this at the start of a research task to check what is already known.
    Results are ranked by semantic similarity.

    Args:
        query: Natural-language query describing what you want to recall.
        max_results: How many findings to return.
    """
    results = memory_client.search(
        query,
        user_id=USER_ID,
        agent_id=AGENT_ID,
        exact_user_match=True,
        exact_agent_match=True,
        max_results=max_results,
        record_types=["memory"],
    )
    if not results:
        return "(no prior durable findings)"
    return "\n".join(
        f"- {r.content}  [distance={r.distance:.3f}]" for r in results
    )


print("Tools defined: tavily_search, save_research_finding, recall_research_findings")

Tools defined: tavily_search, save_research_finding, recall_research_findings


## 7. 构建研究智能体

智能体的 `instructions` 就是它的系统提示词——用来编码你期望行为的地方。对于深度研究智能体，指令应强调：

1. **先回忆再研究**：在访问网络之前先检查长期记忆
2. **注明来源**：使发现可追溯
3. **保存值得记住的内容**：显式提交持久结论

> **💡 关键洞察：** 深度研究智能体的输出质量在很大程度上取决于其指令能否清楚区分*研究*（检索 + 综合）与*对话*（回应 + 确认）。请在系统提示词中编码这一区别。

In [7]:
from agents import Agent, Runner

INSTRUCTIONS = """You are a deep-research agent specialising in human genome exploration.

For every research question:
1. FIRST call `recall_research_findings` to check what is already known from prior sessions.
2. If the prior findings are insufficient or outdated, call `tavily_search` for current sources.
3. Synthesise a clear, cited answer. Prefer authoritative sources (NCBI, OMIM, PubMed).
4. Call `save_research_finding` for each durable conclusion worth remembering across sessions.
   Save one finding per call; keep findings atomic and one sentence long.
5. Present the final answer to the user with inline citations (URLs).

Do not save conversational acknowledgements as findings. Only save factual conclusions.
"""

research_agent = Agent(
    name="GenomeDeepResearcher",
    instructions=INSTRUCTIONS,
    tools=[tavily_search, save_research_finding, recall_research_findings],
    model=MODEL,
)
print(f"Agent created: {research_agent.name}")

Agent created: GenomeDeepResearcher


## 8. 运行研究会话

我们创建一个会话（与记忆存储中的 `thread_id` 关联），并让智能体依次处理一系列研究问题。由于我们传入了 `session=session`，SDK 会在每轮开始时自动注入之前的对话条目，并在结束时持久化新条目。

> **📌 关注点分离。**
> - `Session` 持久化**原始对话**（工作记忆）。
> - 智能体的 `save_research_finding` 工具持久化**持久发现**（长期记忆）。
> 两者存放在同一个 Oracle 数据库中，但通过不同的接口访问。

由于本示例使用实时网页搜索和模型生成，确切的措辞、引用和排序在不同运行之间可能有所不同。需要验证的重要行为是：智能体在研究之前先回忆持久发现、注明来源，并通过记忆工具显式保存新的发现。


In [8]:
from IPython.display import Markdown, display


def _notebook_text(text: str) -> str:
    """Normalize model text so GitHub, VS Code, and terminals render it consistently."""
    return (
        text.replace("\u2013", "-")
        .replace("\u2014", "-")
        .replace("\u2019", "'")
        .replace("\u201c", '"')
        .replace("\u201d", '"')
    )


SESSION_ID = f"genome-session-{RUN_ID}-001"
session = OracleAgentMemorySession(
    session_id=SESSION_ID,
    client=memory_client,
    user_id=USER_ID,
    agent_id=AGENT_ID,
)

research_questions = [
    "What is BRCA1 and what is its primary function in DNA repair?",
    "What is the typical lifetime breast cancer risk associated with pathogenic BRCA1 variants?",
    "How does BRCA1 interact with BRCA2 in homologous recombination?",
]

session_sections = []
for i, q in enumerate(research_questions, 1):
    result = await Runner.run(research_agent, q, session=session)
    session_sections.append(f"### Q{i}: {q}\n\n{_notebook_text(result.final_output.strip())}")

display(Markdown("\n\n---\n\n".join(session_sections)))


### Q1: What is BRCA1 and what is its primary function in DNA repair?

BRCA1 is a human tumor suppressor gene that encodes a nuclear protein important for maintaining genomic stability [(NCBI Gene)](https://www.ncbi.nlm.nih.gov/datasets/gene/672).

Its primary DNA-repair function is to help repair DNA double-strand breaks through homologous recombination, a high-fidelity repair pathway. BRCA1 helps coordinate this process by acting at sites of DNA damage and supporting RAD51-mediated repair together with BRCA2 [(NCBI GeneReviews)](https://www.ncbi.nlm.nih.gov/books/NBK1247) [(NCBI Gene)](https://www.ncbi.nlm.nih.gov/datasets/gene/672).

---

### Q2: What is the typical lifetime breast cancer risk associated with pathogenic BRCA1 variants?

The typical estimate is about 55%-72% lifetime breast cancer risk for women with a pathogenic germline BRCA1 variant by age 70 [(NCBI GeneReviews)](https://www.ncbi.nlm.nih.gov/books/NBK1247).

For context, this is much higher than the roughly 12% general-population lifetime risk cited in the same source [(NCBI GeneReviews)](https://www.ncbi.nlm.nih.gov/books/NBK1247).

---

### Q3: How does BRCA1 interact with BRCA2 in homologous recombination?

BRCA1 and BRCA2 act in the same homologous recombination pathway, but they do different jobs.

BRCA1 acts earlier, helping organize the DNA-damage response and promote repair at double-strand breaks. BRCA2 acts later, mainly by controlling RAD51 loading onto single-stranded DNA so RAD51 can form the repair filament needed for strand invasion [(NCBI GeneReviews)](https://www.ncbi.nlm.nih.gov/books/NBK1247).

A key connection between them is PALB2, which physically links BRCA1 and BRCA2. Through PALB2, BRCA1 helps recruit or stabilize the BRCA2-RAD51 machinery at DNA damage sites, allowing efficient homologous recombination [(Nature/PMC review)](https://pmc.ncbi.nlm.nih.gov/articles/PMC4972490) [(PNAS)](https://www.pnas.org/doi/10.1073/pnas.0811159106).

So, in short: BRCA1 helps set up homologous recombination, and BRCA2 helps execute RAD51-dependent repair, with PALB2 bridging the two [(NCBI GeneReviews)](https://www.ncbi.nlm.nih.gov/books/NBK1247) [(PMC review)](https://pmc.ncbi.nlm.nih.gov/articles/PMC4972490).

## 9. 检查智能体记住了什么

此时，智能体已经积累了两种不同类型的记录：

- **短期会话条目**：为当前会话回放对话的原始 OpenAI Agents SDK 条目。
- **持久研究发现**：智能体通过 `save_research_finding` 有意保存的简洁结论。

这两个列表看起来应该不同。会话条目可能包括用户消息、工具调用和工具输出。持久发现则应是简短、基于事实、可复用的研究笔记——而不是原始的 `call_id` JSON 或完整的工具输出。

In [9]:
# Short-term: conversation items replayed each turn.
session_items = await session.get_items()

# Long-term: durable findings the agent deliberately chose to save.
findings = await memory_client.search_async(
    "BRCA1 function and risk",
    user_id=USER_ID,
    agent_id=AGENT_ID,
    exact_user_match=True,
    exact_agent_match=True,
    max_results=10,
    record_types=["memory"],
)


def _split_finding(content: str) -> tuple[str, str]:
    content = _notebook_text(str(content)).strip()
    if content.startswith("[") and "]" in content:
        topic, finding = content[1:].split("]", 1)
        return topic.strip(), finding.strip()
    return "Research finding", content


raw_json_leaks = [r for r in findings if str(r.content).lstrip().startswith("{")]
quality_note = (
    "Clean saved memories only; raw tool or session JSON was not saved."
    if not raw_json_leaks
    else "Review needed: raw tool or session JSON was saved."
)

summary = (
    "### Memory inspection\n\n"
    f"- Short-term session items stored: `{len(session_items)}`\n"
    f"- Durable research findings saved: `{len(findings)}`\n"
    f"- Stored content quality: {quality_note}\n"
)

rows = ["| Topic | Saved finding |", "|---|---|"]
for record in findings:
    topic, finding = _split_finding(record.content)
    rows.append(f"| {topic} | {finding} |")

display(Markdown(summary + "\n" + "\n".join(rows)))


### Memory inspection

- Short-term session items stored: `32`
- Durable research findings saved: `5`
- Stored content quality: Clean saved memories only; raw tool or session JSON was not saved.

| Topic | Saved finding |
|---|---|
| BRCA1 | BRCA1 encodes a tumor suppressor protein that helps maintain genomic stability and is essential for homologous recombination repair of DNA double-strand breaks. |
| BRCA1 | Women with a germline pathogenic BRCA1 variant have an estimated 55% to 72% lifetime risk of breast cancer by age 70. |
| BRCA1 | BRCA1 promotes DNA repair in part by localizing with BRCA2 and RAD51 at DNA damage sites to support RAD51-mediated repair. |
| BRCA1_BRCA2 | In homologous recombination, BRCA1 helps recruit the PALB2-BRCA2 complex to DNA damage sites, linking DNA end processing to RAD51 loading. |
| PALB2 | PALB2 physically links BRCA1 and BRCA2 during homologous recombination by binding both proteins and facilitating BRCA2-RAD51 repair complex assembly. |

## 10. 验证连续性——使用相同的记忆存储恢复一个新会话

检验具备记忆能力的智能体的真正标准，是它能否从之前会话中断的地方继续。我们来创建一个*新*会话（模拟一个独立进程或之后的某一天），并提出一个基于先前持久发现的问题。

如果智能体能够在不依赖上一个会话记录的情况下回忆起上次运行中关于 BRCA1 的发现，我们就展示了端到端的持久记忆连续性。

这是一个用于教学的基因组学示例，不构成医疗建议。后续问题要求智能体在注明来源的前提下总结有指南支撑的讨论主题，而不是提供针对特定患者的临床建议。


In [10]:
# Simulate a fresh session (new session_id, but same run-scoped user/agent).
FRESH_SESSION_ID = f"genome-session-{RUN_ID}-002"
fresh_session = OracleAgentMemorySession(
    session_id=FRESH_SESSION_ID,
    client=memory_client,
    user_id=USER_ID,
    agent_id=AGENT_ID,
)

followup = (
    "Based on the saved BRCA1 findings, list exactly three concise, "
    "guideline-backed topics a clinician might discuss when explaining "
    "screening considerations for pathogenic BRCA1 variants. Keep this "
    "educational, not patient-specific medical advice. Use plain ASCII "
    "punctuation. Include concise inline URL citations for any sources you mention."
)

result = await Runner.run(research_agent, followup, session=fresh_session)
answer = _notebook_text(result.final_output.strip())
display(
    Markdown(
        "### Fresh session follow-up\n\n"
        f"**Question:** {_notebook_text(followup)}\n\n"
        f"{answer}"
    )
)


### Fresh session follow-up

**Question:** Based on the saved BRCA1 findings, list exactly three concise, guideline-backed topics a clinician might discuss when explaining screening considerations for pathogenic BRCA1 variants. Keep this educational, not patient-specific medical advice. Use plain ASCII punctuation. Include concise inline URL citations for any sources you mention.

1. Breast imaging timing and modality: discuss that guidelines generally recommend earlier, intensified breast screening, typically annual MRI starting by age 25 and annual mammography added around age 30 for BRCA1 carriers (https://www.cancer.gov/types/breast/hp/breast-ovarian-genetics-pdq, https://www.sgo.org/wp-content/uploads/2012/09/PB-182.pdf).

2. Limits of ovarian screening: explain that routine ovarian surveillance with transvaginal ultrasound or CA-125 has not shown clear benefit and is generally not recommended as an effective screening strategy (https://www.cancer.gov/types/breast/hp/breast-ovarian-genetics-pdq, https://www.ncbi.nlm.nih.gov/books/NBK589498).

3. Role of risk-reducing surgery in "screening considerations": note that counseling often includes the option of risk-reducing salpingo-oophorectomy, commonly discussed around ages 35 to 40 after childbearing, because screening alone does not adequately address ovarian cancer risk (https://www.sgo.org/wp-content/uploads/2012/09/PB-182.pdf, https://www.cancer.gov/types/breast/hp/breast-ovarian-genetics-pdq).

## 11. 清理（可选）

清理代码被有意注释掉了。只有当你想从演示 schema 中删除当前运行的会话条目和持久发现时，才取消注释。

上面的设置使用了 `CREATE_IF_NECESSARY`，因此默认情况下重新运行 notebook 不会删除并重建 schema。每次运行都使用唯一的 `RUN_ID`，以避免将当前结果与过时的演示记忆混在一起。

In [11]:
# await session.clear_session()
# await fresh_session.clear_session()
# for r in await memory_client.search_async("BRCA", user_id=USER_ID, agent_id=AGENT_ID, exact_user_match=True, exact_agent_match=True, max_results=100, record_types=["memory"]):
#     memory_client.delete_memory(r.record.id)
# print("Cleaned up current run.")

connection.close()
print("Connection closed.")

Connection closed.


## 关键要点

> **🎯 1. `Session` 协议是清晰的集成点。** OpenAI Agents SDK 只需要四个异步方法（`get_items`、`add_items`、`pop_item`、`clear_session`）即可接入自定义会话后端。你无需修改 runner——只需实现该协议，并通过 `session=` 传入实例。

> **🎯 2. 会话持久化并不等同于长期记忆。** 会话负责为当前智能体循环进行短期对话回放。持久发现应是通过显式工具写入的简洁结论，而不是把原始对话或工具输出一股脑倒进长期记忆。

> **🎯 3. 指令引导记忆的使用。** 必须明确告诉深度研究智能体要*先回忆再研究*并*保存持久结论*。否则 LLM 会把记忆当作可有可无的点缀，存储也会被低价值内容填满。

> **🎯 4. 使用安全的 schema 默认值。** 对于演示和开发，`CREATE_IF_NECESSARY` 比破坏性的重建策略更安全。破坏性的 schema 重置路径应只用于隔离的演示 schema。

> **🎯 5. Oracle AI Database 为两个层级提供同一个受管控的底座。** 短期会话条目和持久发现可以存放在同一个后端中，同时仍是具有不同生命周期的独立记录类型。

> **🎯 6. 连续性是可测试的。** 使用新的 `session_id` 但相同的 `user_id` / `agent_id`，应能得到一个基于先前持久发现进行推理的智能体。这是对具备记忆能力的智能体最简单的端到端测试。